# 02 — NBA Player Data Collection, 2024-25 Season

Pulls every variable in Table 1 of Lautier et al. from the NBA Stats API using `nba_api`. The fetch functions live in `src/nba_roi/collect_stats.py`; the whole step can also be run with `python scripts/fetch_data.py --stats` (about 10 minutes, mostly the per-player game logs).

| Paper variable | Source endpoint |
|---|---|
| FG2O, FG2X, FG3O, FG3X, FTMO, FTMX, PF, STL, BLK, TOV | `PlayerGameLog` (per game) |
| OCRB, AORB, DCRB, ADRB | `LeagueDashPtStats` — Rebounding |
| AC2P, C3P, DFGO, DFGX | `LeagueDashPtDefend` — 2 Pointers / 3 Pointers / Overall |
| SAST, CHGD, DBOX | `LeagueHustleStatsPlayer` |
| APM (= PASSES_MADE − POTENTIAL_AST) | `LeagueDashPtStats` — Passing |
| DDIS | `LeagueDashPtStats` — SpeedDistance |
| PFD | `LeagueDashPlayerStats` (season level only) |

In [ ]:
from nba_roi import config
from nba_roi import collect_stats as cs

---
## 1. Game-by-Game Box Score Stats

In [ ]:
player_list_df = cs.get_player_list()
print(f"Total players: {len(player_list_df)}")
player_list_df.head()

In [ ]:
# Fetch game-by-game log for every player (0.6 s pause between requests)
gamelogs_df = cs.get_game_logs(player_list_df)
print(f"Total game-player rows: {len(gamelogs_df):,}")
gamelogs_df.head()

---
## 2. Rebounding Tracking Stats
Contested vs. uncontested offensive and defensive rebounds (OCRB, AORB, DCRB, ADRB)

In [ ]:
reb_off, reb_def = cs.get_rebounding()
reb_off.head()

---
## 3. Defensive Tracking Stats
Shots defended (DFGO, DFGX), 2-pt contests (AC2P), 3-pt contests (C3P)

In [ ]:
def_2pt = cs.get_defense_2pt()
def_3pt = cs.get_defense_3pt()
def_overall = cs.get_defense_overall()
def_overall.head()

---
## 4. Hustle Stats
Screen assists (SAST), charges drawn (CHGD), defensive box outs (DBOX)

In [ ]:
cs.get_hustle().head()

---
## 5. Passing Stats
Non-assist passes made (APM = PASSES_MADE - POTENTIAL_AST)

In [ ]:
cs.get_passing().head()

---
## 6. Speed & Distance Stats
Defensive distance traveled (DDIS)

In [ ]:
cs.get_speed_distance().head()

---
## 7. Merge All Advanced Stats + Season Standard Stats
`get_advanced_season()` re-fetches each tracking table above and left-joins them onto the season per-game standard stats by `PLAYER_ID`.

In [ ]:
advanced_season = cs.get_advanced_season()
print(f"Final shape: {advanced_season.shape}")
missing = cs.check_table1_vars(advanced_season)
print('All Table 1 variables present.' if not missing else f'MISSING: {missing}')

---
## 8. Save to Excel

In [ ]:
config.RAW_DIR.mkdir(parents=True, exist_ok=True)
gamelogs_df.to_excel(config.GAMELOGS_FILE, index=False)
advanced_season.to_excel(config.ADVANCED_FILE, index=False)
print(f"Saved: {config.GAMELOGS_FILE.name}  ({len(gamelogs_df):,} rows)")
print(f"Saved: {config.ADVANCED_FILE.name}  ({len(advanced_season)} players)")